# 🧠 MLP Backpropagation from First Principles

In this notebook, we implement a Multi-Layer Perceptron (MLP) using only NumPy. We focus on the mathematical derivation of the **Chain Rule** and its manual implementation in code.

## 1. The Theory

We have a network with layers $l=0, \dots, L$.

**Forward Pass**:
- $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$
- $A^{[l]} = g(Z^{[l]})$

**Loss (Cross-Entropy)**:
- $J = -\frac{1}{m} \sum y \log(\hat{y})$

**Backward Pass (Gradients)**:
- $dZ^{[L]} = A^{[L]} - Y$
- $dW^{[l]} = \frac{1}{m} dZ^{(l)} (A^{(l-1)})^T$
- $db^{[l]} = \frac{1}{m} \sum dZ^{(l)}$
- $dZ^{[l-1]} = (W^{[l]})^T dZ^{[l]} \odot g'(Z^{[l-1]})$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

class SimpleMLP:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.b2 = np.zeros((1, output_dim))
        
    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = relu(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = softmax(self.z2)
        return self.a2
        
    def backward(self, X, y, learning_rate):
        m = X.shape[0]
        # Grad w.r.t z2 (output error)
        dz2 = self.a2 - y
        dW2 = (1/m) * np.dot(self.a1.T, dz2)
        db2 = (1/m) * np.sum(dz2, axis=0, keepdims=True)
        
        # Grad w.r.t z1 (hidden layer error)
        dz1 = np.dot(dz2, self.W2.T) * relu_derivative(self.z1)
        dW1 = (1/m) * np.dot(X.T, dz1)
        db1 = (1/m) * np.sum(dz1, axis=0, keepdims=True)
        
        # Update
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1

## 2. Training on XOR Problem

The XOR problem is classic because it requires a non-linear hidden layer.

In [ ]:
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([[1,0], [0,1], [0,1], [1,0]]) # One-hot encoded [1,0] for 0, [0,1] for 1

model = SimpleMLP(2, 4, 2)
losses = []
for i in range(10000):
    out = model.forward(X)
    loss = -np.mean(np.sum(y * np.log(out + 1e-8), axis=1))
    losses.append(loss)
    model.backward(X, y, 0.1)

plt.plot(losses)
plt.title('Training Loss (XOR)')
plt.show()

print(f'Final Predictions:\n{model.forward(X)}')